[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Project Layout &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the folders, `run` and `pip`. The cells after it write the project as the
notebook's worked examples left it, and the last of them installs it into `scratch/stations/.venv`
in editable mode, which downloads setuptools and pytest and takes a few seconds. Run them first,
then the tasks in order, since task 5 tests the function task 4 adds. The last cell removes the
scratch folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
import tomllib
import zipfile
from pathlib import Path

SCRATCH = Path("scratch")
PROJECT = SCRATCH / "stations"
for folder in [PROJECT / "src" / "stations", PROJECT / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def run(*command, folder=SCRATCH):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


def pip(environment, *arguments, folder=SCRATCH):
    """Run this notebook's pip for the Python in an environment: python -m pip --python ENVIRONMENT ..."""
    return run(sys.executable, "-m", "pip", "--python", environment, "--disable-pip-version-check", "--no-color",
               *arguments, folder=folder)


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/src/stations/__init__.py
"""Mean temperatures from the weather stations' readings."""


Writing scratch/stations/src/stations/__init__.py


In [3]:
%%writefile scratch/stations/src/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None."""
    station, celsius = line.strip().split(",")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if line.strip():
            station, celsius = parse_reading(line)
            by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


def to_kelvin(celsius):
    """A temperature in degrees Celsius, in kelvin."""
    return celsius + 273.15


Writing scratch/stations/src/stations/readings.py


In [4]:
%%writefile scratch/stations/src/stations/summary.py
"""Print each station's mean temperature from a file of readings: stations-summary readings.csv"""

import sys

from stations.readings import summarize, to_fahrenheit


def main(arguments=None):
    arguments = sys.argv[1:] if arguments is None else arguments
    with open(arguments[0], encoding="utf-8") as file:
        for station, celsius in summarize(file).items():
            if celsius is None:
                print(station, "no readings")
            else:
                print(station, f"{celsius:.1f} C, {to_fahrenheit(celsius):.1f} F")


Writing scratch/stations/src/stations/summary.py


In [5]:
%%writefile scratch/stations/tests/test_readings.py
from stations.readings import mean, summarize, to_fahrenheit


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


def test_a_blank_line_is_skipped():
    assert summarize(["Bergen,4.2", "", "Bergen,5.8"]) == {"Bergen": 5.0}


Writing scratch/stations/tests/test_readings.py


In [6]:
%%writefile scratch/stations/pyproject.toml
[build-system]
requires = ["setuptools>=77.0.3"]
build-backend = "setuptools.build_meta"

[project]
name = "stations"
version = "0.1.0"
description = "Mean temperatures from the weather stations' readings"
requires-python = ">=3.10"
dependencies = []

[project.optional-dependencies]
test = ["pytest==8.4.2"]

[project.scripts]
stations-summary = "stations.summary:main"

[tool.pytest.ini_options]
testpaths = ["tests"]


Writing scratch/stations/pyproject.toml


In [7]:
%%writefile scratch/stations/.gitignore
.venv/
build/
dist/
*.egg-info/
.pytest_cache/


Writing scratch/stations/.gitignore


In [8]:
run(sys.executable, "-m", "venv", "--without-pip", "stations/.venv")
code, printed = pip(".venv", "install", "-q", "-e", ".[test]", folder=PROJECT)

print("install exit code:", code)


install exit code: 0


**1.** The project's own files.


In [9]:
made = {".venv", "build", "dist", "stations.egg-info", ".pytest_cache"}
files = [path.relative_to(PROJECT) for path in PROJECT.rglob("*") if path.is_file()]

print(sorted(str(path) for path in files if not made & set(path.parts)))


['.gitignore', 'pyproject.toml', 'src/stations/__init__.py', 'src/stations/readings.py', 'src/stations/summary.py', 'tests/test_readings.py']


`made & set(path.parts)` is empty for a file with none of those folders anywhere in its path, which
is the test for a file the project keeps.


**2.** The record, read with tomllib.


In [10]:
record = tomllib.loads((PROJECT / "pyproject.toml").read_text())

print(record["project"]["name"], record["project"]["version"])
print("stations-summary calls", record["project"]["scripts"]["stations-summary"])


stations 0.1.0
stations-summary calls stations.summary:main


`stations.summary:main` names the module, then a colon, then the function in it.


**3.** An import from outside the project.


In [11]:
importing = "from stations.readings import to_fahrenheit; print(to_fahrenheit(-40))"
code, printed = run("stations/.venv/bin/python", "-c", importing)

print("exit code:", code, "|", printed)


exit code: 0 | -40.0


The command ran in `scratch`, and the environment's Python found the installed package.


**4.** A new function, with no install.


In [12]:
addition = ('\n\ndef to_celsius(fahrenheit):\n'
            '    """A temperature in degrees Fahrenheit, in degrees Celsius."""\n'
            '    return (fahrenheit - 32) * 5 / 9\n')
readings = PROJECT / "src" / "stations" / "readings.py"
readings.write_text(readings.read_text() + addition)

code, printed = run("stations/.venv/bin/python", "-c", "from stations.readings import to_celsius; print(to_celsius(212))")
print("exit code:", code, "|", printed)


exit code: 0 | 100.0


The editable install reads `src/stations/readings.py` as it is now, so the new function imports at
once.


**5.** A test of it, run from outside.


In [13]:
tests = PROJECT / "tests" / "test_readings.py"
source = tests.read_text().replace("import mean,", "import mean, to_celsius,")
tests.write_text(source + "\n\ndef test_boiling_point_in_celsius():\n    assert to_celsius(212) == 100\n")

code, printed = run("stations/.venv/bin/pytest", "-q", "--no-header", "stations")
print("exit code:", code)
print(printed)


exit code: 0
....                                                                     [100%]
4 passed


The `pytest` script, run from `scratch` with the project's folder as its argument, found the tests
and imported the installed package, which the flat project's tests could not do.


**6.** A wheel, and what is inside.


In [14]:
code, printed = run(sys.executable, "-m", "pip", "wheel", "--disable-pip-version-check", "-q", "--no-deps",
                    "-w", SCRATCH.resolve() / "task-dist", ".", folder=PROJECT)
print("build exit code:", code)

wheel = next((SCRATCH / "task-dist").glob("*.whl"))
with zipfile.ZipFile(wheel) as archive:
    print(wheel.name, [name for name in archive.namelist() if ".dist-info/" not in name])
    print("to_celsius in the wheel:", b"def to_celsius" in archive.read("stations/readings.py"))


build exit code: 0
stations-0.1.0-py3-none-any.whl ['stations/__init__.py', 'stations/readings.py', 'stations/summary.py']
to_celsius in the wheel: True


The package's three files, with `to_celsius` in `readings.py`, since a wheel is built from the files
as they are.

Last, remove the scratch folder:


In [15]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Project Layout](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/09-project-layout.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
